# AttritionIQ: Exploratory Data Analysis & Feature Engineering

This notebook demonstrates exploratory data analysis, data sanitization, domain feature engineering, and baseline correlation analysis on the historical employee attrition dataset.

In [ ]:
import pandas as pd
import numpy as np

# 1. Load historical employee dataset
df = pd.read_csv('../data/raw/HR_Employee_Data.csv')
print('Dataset Shape:', df.shape)
df.head()

## 2. Data Cleaning & Type Conversion

Converting percentage strings (`satisfaction_level` and `last_evaluation`) into standardized float proportions [0.0, 1.0].

In [ ]:
def clean_pct(val):
    if isinstance(val, str) and '%' in val:
        return float(val.replace('%', '')) / 100.0
    return float(val)

df['satisfaction_level'] = df['satisfaction_level'].apply(clean_pct)
df['last_evaluation'] = df['last_evaluation'].apply(clean_pct)
df.info()

## 3. Exploratory Analysis

Examining target balance and key correlations.

In [ ]:
attrition_counts = df['left'].value_counts()
print('Attrition Distribution:')
print(attrition_counts)
print(f'Historical Attrition Rate: {df["left"].mean():.2%}')

## 4. Domain Feature Engineering

Formulating sharp domain parameters: evaluation-satisfaction difference, burnout risk, overworked indicator, and salary-to-workload ratio.

In [ ]:
df['eval_sat_diff'] = df['last_evaluation'] - df['satisfaction_level']
df['hours_per_project'] = df['average_montly_hours'] / df['number_project']
df['burnout_risk'] = ((df['last_evaluation'] > 0.75) & (df['average_montly_hours'] > 220) & (df['satisfaction_level'] < 0.45)).astype(int)
df['overworked'] = ((df['average_montly_hours'] > 240) & (df['number_project'] >= 5)).astype(int)

print('Feature comparison by Attrition Status:')
df.groupby('left')[['eval_sat_diff', 'hours_per_project', 'burnout_risk', 'overworked']].mean()